In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import sklearn
import torch
from torchvision import transforms
#from google.colab import ai

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
colab_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/dataset.csv"
local_path = "../FracAtlas/dataset.csv"
df = pl.read_csv(local_path)
df.head()


image_id,hand,leg,hip,shoulder,mixed,hardware,multiscan,fractured,fracture_count,frontal,lateral,oblique
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""IMG0000000.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000001.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000002.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000003.jpg""",0,1,0,0,0,0,1,0,0,0,1,1
"""IMG0000004.jpg""",0,1,0,0,0,0,1,0,0,0,1,1


In [ ]:
fractured = df.filter(df["fractured"] == 1)
nonfractured = df.filter(df["fractured"] == 0)

train, test_validate = sklearn.model_selection.train_test_split(df, stratify=df["fractured"], train_size=0.7, test_size=0.3, random_state=42)
test, validation = sklearn.model_selection.train_test_split(test_validate, stratify=test_validate["fractured"], train_size=0.5, test_size=0.5, random_state=42)

fractured_local_path = "../FracAtlas/images/Fractured/"
non_fractured_local_path = "../FracAtlas/images/Non_fractured/"

fractured_colab_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Fractured/"
non_fractured_colab_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Non_fractured/"

train_paths = train.with_columns(
    pl.when(pl.col("fractured") == 1).then(fractured_local_path + pl.col("image_id")).otherwise(non_fractured_local_path + pl.col("image_id")).alias("paths")
)

test_paths = test.with_columns(
    pl.when(pl.col("fractured") == 1).then(fractured_local_path + pl.col("image_id")).otherwise(non_fractured_local_path + pl.col("image_id")).alias("paths")
)

validation_paths = validation.with_columns(
    pl.when(pl.col("fractured") == 1).then(fractured_local_path + pl.col("image_id")).otherwise(non_fractured_local_path + pl.col("image_id")).alias("paths")
)

train_paths = train_paths["paths"]
test_paths = test_paths["paths"]
validation_paths = validation_paths["paths"]

train_labels = train["fractured"]
test_labels = test["fractured"]
validation_labels = validation["fractured"]




In [36]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                          
    transforms.Normalize([0.485, 0.456, 0.406],    
                         [0.229, 0.224, 0.225])   
])


In [ ]:
from torch.utils.data import Dataset 
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

class FractureDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self,index):
        img = Image.open(self.paths[index]).convert("RGB")
        img = self.transform(img)
        label = self.labels[index]
        return img, label

In [ ]:
from torch.utils.data import DataLoader

train_dataset = FractureDataset(train_paths.to_list(), train_labels.to_list(), transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = FractureDataset(validation_paths.to_list(), validation_labels.to_list(), transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

test_dataset = FractureDataset(test_paths.to_list(), test_labels.to_list(), transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 1) 
)
models.MobileNet_V2_Weights.IMAGENET1K_V1.transforms

In [86]:
train_fractured_count = train["fractured"] == 1
true, false = train_fractured_count.value_counts()
print(true, false)

shape: (2,)
Series: 'fractured' [bool]
[
	false
	true
] shape: (2,)
Series: 'count' [u32]
[
	2356
	502
]


In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# pos_weight accounts for class imbalance (~4.7:1 non-fractured to fractured)
pos_weight = torch.tensor([2356 / 502]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Only train the classifier head — base is frozen
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

Base Model

In [40]:
num_epochs = 10


for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    total_train_batches = len(train_loader)

    for batch_idx, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(device)
        labels = labels.float().to(device)  

        optimizer.zero_grad()
        outputs = model(images).squeeze(1)   
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step() 

        running_loss += loss.item() * images.size(0)
        preds = (outputs > 0).long()         
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

        # Print periodic batch updates so training does not look stuck
        if batch_idx % 20 == 0 or batch_idx == total_train_batches:
            print(f"Epoch {epoch+1}/{num_epochs} | Train Batch {batch_idx}/{total_train_batches} | "
                  f"Batch Loss: {loss.item():.4f} | Running Acc: {correct/total:.4f}")

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    total_val_batches = len(val_loader)

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(val_loader, start=1):
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = (outputs > 0).long()
            val_correct += (preds == labels.long()).sum().item()
            val_total += labels.size(0)

            if batch_idx % 10 == 0 or batch_idx == total_val_batches:
                print(f"Epoch {epoch+1}/{num_epochs} | Val Batch {batch_idx}/{total_val_batches} | "
                      f"Batch Loss: {loss.item():.4f} | Running Acc: {val_correct/val_total:.4f}")

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

Epoch [1/10] Train Loss: 1.0334  Train Acc: 0.6417  Val Loss: 0.9196  Val Acc: 0.7781
Epoch [2/10] Train Loss: 0.8900  Train Acc: 0.7218  Val Loss: 0.8576  Val Acc: 0.7341
Epoch [3/10] Train Loss: 0.8153  Train Acc: 0.7687  Val Loss: 0.8390  Val Acc: 0.7357
Epoch [4/10] Train Loss: 0.7886  Train Acc: 0.7694  Val Loss: 0.8569  Val Acc: 0.8206
Epoch [5/10] Train Loss: 0.8053  Train Acc: 0.7680  Val Loss: 0.9309  Val Acc: 0.8613
Epoch [6/10] Train Loss: 0.7913  Train Acc: 0.7740  Val Loss: 0.8171  Val Acc: 0.7423
Epoch [7/10] Train Loss: 0.7736  Train Acc: 0.7841  Val Loss: 0.8219  Val Acc: 0.7781
Epoch [8/10] Train Loss: 0.7466  Train Acc: 0.8051  Val Loss: 0.8111  Val Acc: 0.7586
Epoch [9/10] Train Loss: 0.7904  Train Acc: 0.7523  Val Loss: 0.8075  Val Acc: 0.7798
Epoch [10/10] Train Loss: 0.7622  Train Acc: 0.7880  Val Loss: 0.8221  Val Acc: 0.7602


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                          
    transforms.Normalize([0.485, 0.456, 0.406],    
                         [0.229, 0.224, 0.225])   
])


In [ ]:
torch.save(model.state_dict(), 'model.pth')
print('Model saved!')